# 第 6 章 · 记忆管理与上下文压缩

> 学习笔记 · 源自《Harness Engineering》配套代码仓库 [harness-py-book](https://github.com/Luxuzhou/harness-py-book)

**一句话总纲**：上下文窗口是稀缺资源——它会被噪声填满（**腐烂**）、要按预算分配（**五区划分**）、超支后要分级压缩（**四级火箭**）；真正的长期记忆放在窗口之外（**六层记忆架构** + **JSONL 会话持久化**），并在下次会话开始时被"唤醒"（**三步唤醒流程**）。

## 本章导览

| 小节 | 主题 | 核心机制 | 配图 |
|---|---|---|---|
| 6.1 | 上下文腐烂 | 有效窗口 < 物理窗口；3 层损耗；4 种症状 | 图 6-1 |
| 6.2 | 渐进式有损压缩 | L0 截断 → L1 缩略 → L2 摘要 → L3 应急 | 图 6-2 |
| 6.3 | 压缩触发与分级响应 | Token 预算五区；压力阈值决策链 | 图 6-3 / 6-4 |
| 6.4 | 跨会话长期记忆 | 六层记忆架构；Auto Memory；Dream 整理 | 表 6-6 / 6-7 |
| 6.5 | 会话持久化与断点续传 | JSONL / WAL；SessionWriter；三步唤醒 | 表 6-8 |

**本章插图**（复现自书中图 6-1 ~ 6-4）：

```text
src/context-rot-curve.svg        图 6-1  上下文腐烂曲线
src/four-stage-compression.svg   图 6-2  四级压缩阶梯模型
src/token-budget-zones.svg       图 6-3  Token 预算五区
src/pressure-response-flow.svg   图 6-4  压力响应决策链
```


<div style="text-align:center"><img src="../src/context-rot-curve.svg" width="820"/></div>

## 6.1 上下文腐烂：为什么"窗口大"不等于"记得住"

**上下文腐烂（Context Rot）**：随着对话轮次增加，即使上下文窗口还有大量剩余空间，模型对信息的**有效利用率**也在持续下降——**有效窗口（Effective Window）< 物理窗口**。

### 3 层损耗

1. **物理占用**：旧工具输出原样堆在窗口里，Token 被占满，留给新内容的空间越来越少；
2. **信噪比塌陷**：历史消息里真正有效的信息占比急剧下降——1 轮对话约 1500 Token 有效信息占 5~8%，10 轮后膨胀到约 15000 Token、有效信息却**不足 2%**（表 6-1）；
3. **注意力衰减**：模型对超长上下文中段内容"读不进"（lost in the middle），关键约束被淹没在噪声中。

**表 6-1　对话轮次与有效信息占比（书中实测）**

| 对话轮次 | 上下文 Token（约） | 有效信息占比 |
|---:|---:|---:|
| 1 | 1,500 | 5~8% |
| 3 | 4,500 | ↓ |
| 5 | 7,500 | ↓ |
| 10 | 15,000 | < 2% |

### 腐烂的 4 种典型症状

| 症状 | 表现 |
|---|---|
| **提前终止** | Agent 走到一半宣称"任务完成"，实际只完成了子集 |
| **环境误区** | 忘记早前读过的环境信息，重复犯错、重复探索 |
| **虚标完成** | 报告成功但交付物缺失或损坏——它"以为"自己做完了 |
| **上下文失忆** | 完全忘记几轮前的关键决定，重新讨论已定方案 |


<div style="text-align:center"><img src="../src/four-stage-compression.svg" width="820"/></div>

## 6.2 渐进式有损压缩：四级火箭

**设计哲学**：把"压缩强度"做成阶梯，**先便宜后昂贵**——低强度级别不够用时才升级到高级别，每次只付出当下必要的代价。

**表 6-2　四级压缩对比**

| 级别 | 名称 | 机制 | 成本 | 丢失程度 |
|---|---|---|---|---|
| L0 | 硬性截断 | microcompact：按长度切片，只保留最近 `preserve` 条 | **≈ 0** | 零语义丢失（只丢明显过期的） |
| L1 | 缩略替换 | snip：旧工具输出/长回复压成一行 `[snipped: ...]` | **≈ 0** | 低（保留摘要行可检索） |
| L2 | 语义摘要 | _compact：调模型对中段历史做迭代摘要 | **1 次 API** | 可控（Hermes 6 段式锁住关键信息） |
| L3 | 应急压缩 | reactive：preserve 减半 + 多轮循环压到达标 | **2 次 API** | 高（保进程不保细节） |

### compress 总入口

```python
def compress(self, messages, target_tokens, *, trim_call=None, reactive=False):
    # 依次尝试 L0 → L1 → L2，每级执行后重新计量
    # 未达标才升级到下一级；reactive=True 时启用 L3，至多循环 6 轮
```

关键点：**每级之后都检查是否已达标**，达标即停——避免"过度压缩"白白丢失信息。


### 6.2.1　L0 硬性截断（microcompact）

```python
def _l0_call(...):
    # pass_num 记录当前轮次：先试探性执行，效果不足再升级
    ...

def _microcompact(self, messages, preserve):
    # 只做一件事：把最早的 (preserve*2+1) 条之前的内容切掉
    cutoff = max(0, len(messages) - preserve * 2 - 1)
    return messages[cutoff:]
```

- **零判断开销**：不看内容、不调 API，仅做长度切片；
- `preserve_messages` 默认 **4**：永远保住最近 4 条消息，保证"当前正在做的事"不被误伤；
- 适合腐烂早期：窗口刚超限、噪声还不深时，一刀切掉最老的块往往就够。

### 6.2.2　L1 缩略替换（snip）

```python
def _snip(self, messages):
    for i, m in enumerate(messages):
        if i == 0:            # 跳过系统提示
            continue
        # 不对称阈值：工具结果 200 字符、assistant 回复 600 字符才替换
        limit = 200 if m["role"] == "tool" else 600
        if len(m["content"]) > limit:
            preview = m["content"][:80]
            m["content"] = f"[snipped: {preview}...]"
            snipped += 1
        if snipped >= 3:      # 每轮最多替换 3 条，控制单轮损失
            break
```

设计细节：

- **不对称阈值**：工具结果（200 字符）比助手长回复（600 字符）更容易被压——工具输出往往是最长的噪声源；
- **每轮最多 3 条**：限制单次压缩的信息损失，宁可下一轮再压；
- **保留 80 字符预览**：被压内容仍可辨识（"哪个文件的输出"），而不是彻底消失。

### 6.2.3　L2 语义摘要（_compact）

```python
def _compact(self, messages, ...):
    prefix = messages[:k]                      # 开头保护段
    tail   = messages[-tail_n:]                # 结尾保护段
    candidates = messages[k:-tail_n][:20]      # 待摘要的中段，最多 20 条
    chunks = [c["content"][:300] for c in candidates]   # 每条最多 300 字符

    summary = self._hermes_iterate(chunks)     # Hermes 迭代式摘要
    if len(summary) <= 30:                     # 摘要太短 → 视为失败
        return prefix + tail                   # 回退为"硬截断拼头尾"

    wrapped = f"<system-reminder>Earlier conversation history was compacted.\n{summary}</system-reminder>"
    result = prefix + [{"role": "user", "content": wrapped}] + tail
    # 压缩后重读 CLAUDE.md：项目规则永远在场
    ...
```

四个防御细节：

1. **三段式**（prefix / candidates / tail）：开头系统约束和结尾当前任务永远不进摘要机；
2. **限量投喂**：candidates 最多 20 条、每条 300 字符——喂给摘要模型的输入本身也要有预算；
3. **有效性验证**：摘要 < 30 字符判为失败，**回退**到 prefix+tail 拼接而不是拿垃圾摘要顶上；
4. **压缩后重读 CLAUDE.md**：摘要会不可避免地稀释项目规则，主动重读补回来。


### 🔬 动手实验 1：L0 截断与 L1 缩略替换（最小复现）

运行下面的代码，观察 Token 数在 L0 / L1 两级压缩后的变化——体会"零成本压缩"如何工作。


In [ ]:
# ── 实验 1：L0 microcompact + L1 snip 最小复现 ──
def make_msgs():
    msgs = [{"role": "system", "content": "You are a coding agent."}]
    for i in range(1, 9):
        msgs.append({"role": "user", "content": f"任务 {i}：请读取文件 file_{i}.py"})
        msgs.append({"role": "tool", "content": "x" * 400 + f" [file_{i}.py 的完整内容]"})
        msgs.append({"role": "assistant", "content": "y" * 700 + f" [关于 file_{i}.py 的长结论]"})
    return msgs

def count_tokens(msgs):            # 近似：1 token ≈ 2 字符
    return sum(len(m["content"]) for m in msgs) // 2

def microcompact(msgs, preserve=4):
    # L0：只保最近 preserve*2+1 条，零判断切片
    cutoff = max(0, len(msgs) - preserve * 2 - 1)
    return msgs[cutoff:]

def snip(msgs, tool_limit=200, asst_limit=600, per_round=3):
    # L1：超长旧消息压成一行预览，每轮最多 3 条，不对称阈值
    out, n = [], 0
    for i, m in enumerate(msgs):
        if i == 0:
            out.append(m); continue
        limit = tool_limit if m["role"] == "tool" else asst_limit
        if len(m["content"]) > limit and n < per_round:
            out.append({**m, "content": f"[snipped: {m['content'][:80]}...]"})
            n += 1
        else:
            out.append(m)
    return out

msgs = make_msgs()
print(f"原始   : {len(msgs):2d} 条, {count_tokens(msgs)} tokens")
msgs = microcompact(msgs)
print(f"L0 之后: {len(msgs):2d} 条, {count_tokens(msgs)} tokens  ← 一刀切，零判断")
msgs = snip(msgs)
print(f"L1 之后: {len(msgs):2d} 条, {count_tokens(msgs)} tokens  ← 本轮只压 3 条，保留预览")
print()
print("被替换后的样子：", next(m["content"][:60] for m in msgs if m["content"].startswith("[snipped")))


### 6.2.4　Compact Instructions：防"摘要后忘了目标"

L2 摘要最大的风险是**把任务目标本身也当噪声压掉了**——摘要模型 summaries 得很漂亮，Agent 却忘了自己要干什么。解决办法：给摘要模型一份**紧凑指令（Compact Instructions）**，声明哪些内容**必须原样保留**：

1. **当前任务目标**（原话保留，不做改写）；
2. **已确认的关键约束**（技术栈、路径、验收标准）；
3. **未完成事项与下一步计划**；
4. **用户显式交代的偏好与禁令**。

**表 6-3　紧凑指令对摘要质量的影响（书中实测，10 个恢复任务）**

| 方案 | 摘要字数 | 恢复成功 | 通过率 |
|---|---:|---:|---:|
| without（无紧凑指令） | 439 | 0 / 10 | **0%** |
| with_short（短指令） | 814 | 10 / 10 | **100%** |
| with_annotated（带标注槽式） | 355 | 8 / 10 | **80%** |

书中正文还给出另一组对照：朴素写法通过率仅 **10%**，改为槽式 `[x]` 标记后升至 **60%**。

**两个结论**：

- 紧凑指令让摘要从"自由发挥"变成"填空题"——**结构化约束远胜于描述性叮嘱**；
- 指令本身也要克制：with_annotated 字数少一半，通过率略低于 with_short——**标注规则复杂到一定程度会出现边际递减**。

### 6.2.5　L3 应急压缩（reactive 模式）

```python
# 压力极大时：保护段减半，反复压到达标
preserve = max(preserve // 2, 1)   # 4 → 2 → 1，最少保 1 条
# reactive=True 时 compress 至多循环 6 轮，仍不达标则放弃
```

- **用途**：预算已穿、必须立刻腾出大量空间的兜底场景；
- **代价**：2 次 API + 大量历史进入"保进程、不保细节"的状态；
- **6 轮上限**：防止"压了还不达标 → 再压 → 无限循环"的失控。

### 前置工序：修复孤儿配对

压缩会切断 `tool_call ↔ tool_result` 的对应关系，某些 API（如 OpenAI）会直接报 400。`_fix_orphaned_tool_pairs` 用**两轮扫描**修复：

- 第一轮：登记所有 tool_call 的 id；
- 第二轮：凡 tool_result 找不到对应 call 的，补一个占位 call；凡 call 找不到 result 的，补一个占位 result。

**表 6-4　启用压缩前后对比（书中实测）**

| 配置 | 上下文 Token | 关键信息召回 | 回答质量 Q | 触发压缩次数 |
|---|---:|---:|---:|---:|
| 压缩 off | 9,241 | 16.33 | 4.0 | — |
| 压缩 on | 1,327 | 8.33 | 5.0 | 10.3 次 |

Token 降到 **1/7**，质量反而上升——腐烂清掉后，模型"看得见"的信息变多了。

**表 6-5　各级别实际贡献（触发次数分布，帕累托现象）**

| 级别 | 单场景触发次数 | 说明 |
|---|---|---|
| L0 | 10.5 ~ 17.0 | 绝大多数压缩由 L0 消化 |
| L1 | 2.0 ~ 7.5 | 中度腐烂时接管 |
| L2 | 2.0 ~ 3.0 | 少量深度压缩 |
| L3 | 0 | 几乎不触发——是保险，不是日常 |


## 6.3　压缩触发与分级响应

### Token 预算五区（TokenBudget）

```python
@dataclass(frozen=True)
class TokenBudget:
    total: int

    def allocate(self):
        out  = int(self.total * 0.15)   # 输出预留：防响应被截断
        sys_ = int(self.total * 0.10)   # System 提示词
        mem  = int(self.total * 0.20)   # Memory 记忆束
        tool = int(self.total * 0.05)   # 当前工具结果
        history = max(0, self.total - out - sys_ - mem - tool)  # 剩余全给历史
        return out, sys_, mem, tool, history
```

设计逻辑：**先给"必保项"划地盘，剩余 `max(0, budget)` 全部留给历史区**——历史区是唯一可牺牲的弹性区，四级火箭只作用于它。

<div style="text-align:center"><img src="../src/token-budget-zones.svg" width="820"/></div>

### 压力感知与决策链

```python
def should_compress(self, used_tokens):
    ratio = used_tokens / self.total
    return ratio >= 0.85        # 85% 触发线：留缓冲，等爆了再压就晚了

def decide_compression(self, ratio):
    if   ratio < 0.70: return None               # 健康，不动
    elif ratio < 0.85: return "microcompact"     # L0
    elif ratio < 0.95: return "snip_and_compact" # L1 + L2
    elif ratio < 1.00: return "emergency"        # L3 应急
    else:              return "reactive"         # 极限模式
```

<div style="text-align:center"><img src="../src/pressure-response-flow.svg" width="820"/></div>

**阈值经验值**：0.70 / 0.85 / 0.95 / 1.00 四道阈值把"何时压、压多狠"变成确定性规则——**高级别动作只在低级别不够时才出手**。

**表 6-4 / 6-5 回看**：L0 承担了绝大多数压缩（帕累托分布），L3 几乎从不触发。好的压缩系统不是天天急救，而是**让便宜的手段处理掉 90% 的日常**。


### 🔬 动手实验 2：Token 预算五区 + 压力决策链

复现 `TokenBudget.allocate` 与 `decide_compression`，观察不同用量下决策链如何升级。


In [ ]:
# ── 实验 2：Token 预算五区与压力响应 ──
from dataclasses import dataclass

@dataclass(frozen=True)
class TokenBudget:
    total: int

    def allocate(self):
        out  = int(self.total * 0.15)   # 输出预留
        sys_ = int(self.total * 0.10)   # System 提示词
        mem  = int(self.total * 0.20)   # Memory 记忆束
        tool = int(self.total * 0.05)   # 当前工具结果
        history = max(0, self.total - out - sys_ - mem - tool)  # 剩余全给历史
        return {"output": out, "system": sys_, "memory": mem,
                "tools": tool, "history": history}

    def should_compress(self, used_tokens):
        return used_tokens / self.total >= 0.85

def decide_compression(ratio):
    if   ratio < 0.70: return None
    elif ratio < 0.85: return "microcompact"      # L0
    elif ratio < 0.95: return "snip_and_compact"  # L1 + L2
    elif ratio < 1.00: return "emergency"         # L3
    else:              return "reactive"          # 极限模式

budget = TokenBudget(128_000)
zones = budget.allocate()
print("128K 预算分配：")
for k, v in zones.items():
    print(f"  {k:8} {v:>7,} tokens  ({v/128000:.0%})")

print("\n压力决策演示：")
for used in (88_000, 108_000, 120_000, 126_000, 130_000):
    r = used / 128_000
    act = decide_compression(r)
    print(f"  used={used:>7,}  ratio={r:.2f}  触发压缩={budget.should_compress(used)}  -> {act}")


## 6.4　跨会话长期记忆：六层架构

窗口内的压缩只解决"这一场对话"的问题；**跨会话记忆**靠的是一套分层存储。

**表 6-6　六层记忆架构**

| 层 | 位置 | 生命周期 | 加载方式 |
|---|---|---|---|
| 全局层 | `~/.claude/CLAUDE.md` | 永久 | 每次会话自动注入 |
| 项目层 | `./CLAUDE.md` | 项目级 | 每次会话自动注入 |
| 规则层 | `.claude/rules/*.md` | 项目级 | 按需加载 |
| 会话层 | `harness_sessions/<id>.jsonl` | 单会话 | 恢复时加载 |
| 学习层 | `memory/MEMORY.md` | 跨会话 | 自动加载前 200 行 |
| 当前层 | 上下文窗口 | 实时 | — |

### 写入：write_memory（追加写 + 索引去重）

```python
def write_memory(self, title, content, mem_dir=None):
    # MEMORY.md 前 200 行自动加载、单文件超 5000 字符截断
    filename = slug(title) + ".md"
    line = f"- [{title}]({filename})"
    # 索引行已存在 → 不重复写（去重靠精确匹配索引行）
```

**页表设计模式**：`MEMORY.md` 只存"页表"（一行一个链接），正文分散在 `memory/*.md` 小文件里——

- 自动注入的只有页表（省 Token）；
- Agent 需要某条记忆时按链接**按需读取**详情；
- 类似操作系统：常驻内存的是页表，数据页按需换入。

### 读取：_load_memory_bundle（Hermes 围栏）

```python
bundle = f"<memory-context>\n{memory_md}\n</memory-context>"
```

用 XML 围栏包裹，与正文明确隔离，防止记忆内容被误当成用户指令。

### 项目标识：git rev-parse

Auto Memory 用 `git rev-parse --show-toplevel` 把**仓库根目录**作为项目唯一标识——同一仓库无论从哪个子目录启动，都能找到同一份记忆。

### 整理：Dream 离线整理（睡眠时整理记忆）

```python
def dream(self, mem_dir: Path) -> dict:
    result = {"merged": 0, "fixed": 0, "pruned": 0}
    # 1) 行级去重：norm = line.strip().lower()，重复索引行合并
    # 2) 修复相对日期："昨天""上周" → 绝对日期
    # 3) 清理空文件：正文为空的记忆文件 unlink
    # 4) _rebuild_index()：重建页表
    return result
```

**表 6-7　Dream 整理效果（10 份记忆样例）**

| 指标 | 整理前 | 整理后 | 变化 |
|---|---:|---:|---|
| MEMORY.md 总行数 | 344 | 290 | **-15.7%** |
| long_stream.md | 253 | 203 | 约 -19.8% |

要点：**去重以"行"为单位做归一化比较**（strip + lower），而不是全文哈希——页表行重复是高频病灶。


### 🔬 动手实验 3：JSONL 会话持久化与崩溃安全恢复

复现 SessionWriter 的 uuid 链与"残行跳过"恢复逻辑——人为制造一行坏数据，看恢复是否安然无恙。


In [ ]:
# ── 实验 3：SessionWriter 简化版 + 崩溃恢复 ──
import json, uuid, tempfile
from datetime import datetime
from pathlib import Path

class SessionWriter:
    def __init__(self, session_id, directory):
        self.session_id = session_id
        self.path = Path(directory) / f"{session_id}.jsonl"
        self._prev = None                          # uuid 链头
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._write({"type": "session", "sessionId": session_id})

    def write_message(self, role, content):
        msg_uuid = str(uuid.uuid4())
        event_type = "tool_result" if role == "tool" else role   # 书中的类型转换
        self._write({"type": event_type, "uuid": msg_uuid,
                     "prev_uuid": self._prev,      # 链式指针
                     "content": content,
                     "timestamp": datetime.now().isoformat()})
        self._prev = msg_uuid
        return msg_uuid

    def _write(self, event):                       # 全类唯一碰文件的地方
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, ensure_ascii=False) + "\n")

d = Path(tempfile.mkdtemp())
w = SessionWriter("demo-001", d)
w.write_message("user", "帮我重构 utils.py")
w.write_message("assistant", "好的，我先读取文件……")
w.write_message("tool", "工具输出：文件共 120 行")
w.write_message("assistant", "完成重构，共 3 处修改")

# 模拟崩溃：往尾部追加一行被截断的 JSON
with open(w.path, "a", encoding="utf-8") as f:
    f.write('{"type":"assistant","content":"被截断的残行')

def load_session_messages(path):
    valid = {"user", "assistant", "system", "tool_result"}
    out = []
    for line in open(path, encoding="utf-8"):
        try:
            ev = json.loads(line)
        except json.JSONDecodeError:
            continue                               # 崩溃安全：残行静默跳过
        if ev.get("type") in valid:
            out.append(ev)
    return out

events = load_session_messages(w.path)
print(f"恢复出 {len(events)} 条有效事件（坏行已被跳过）：")
for e in events:
    print(f"  [{e['type']:9}] uuid={e['uuid'][:8]}…  {e['content']}")


### 🔬 动手实验 4：Dream 离线整理之行级去重

复现 Dream 的去重规则：`norm = line.strip().lower()`——体会为什么按"行 + 归一化"去重而不是按全文哈希。


In [ ]:
# ── 实验 4：Dream 行级去重 ──
def dream_dedup(index_lines):
    seen, out = set(), []
    for line in index_lines:
        norm = line.strip().lower()      # 归一化：去首尾空白 + 小写
        if norm and norm not in seen:
            seen.add(norm)
            out.append(line.rstrip())
    return out

index = [
    "- [数据库连接池调优](db-pool.md)",
    "- [数据库连接池调优](db-pool.md)   ",   # 尾随空格 → 应被合并
    "- [数据库连接池调优](DB-POOL.MD)",       # 仅大小写不同 → 应被合并
    "- [重构 utils.py 的三次失败](refactor.md)",
    "",                                       # 空行 → 应被丢弃
    "- [数据库连接池调优](db-pool.md)",       # 完全重复 → 应被合并
]

cleaned = dream_dedup(index)
print(f"{len(index)} 行 → {len(cleaned)} 行")
for l in cleaned:
    print("  保留:", l if l else "(空)")


## 6.5　会话持久化与断点续传

### 6.5.1　JSONL 追加日志（WAL 思想）

会话以 **JSONL**（每行一个 JSON 事件）**追加写**到 `harness_sessions/<session_id>.jsonl`：

- **append-only**：只追加不改写，崩溃时最多丢最后一行（WAL 思想）；
- `ensure_ascii=False`：中文原样存储，文件可读可 diff；
- **崩溃安全**：恢复时逐行 `json.loads`，遇 `JSONDecodeError` 静默跳过残行——**读到哪算哪**，不会因一行坏数据丢掉整个会话。

### SessionWriter：uuid 链

```python
class SessionWriter:
    def __init__(self, session_id, directory, cwd=None):
        ...
        self._prev_uuid = None                # 链头
        directory.mkdir(parents=True, exist_ok=True)
        self._write({"type": "session", "sessionId": session_id, ...})

    def write_message(self, role, content, **extra) -> str:
        event_type = "tool_result" if role == "tool" else role
        msg_uuid = str(_uuid())
        event = {
            "type": event_type,
            "uuid": msg_uuid,
            "prev_uuid": self._prev_uuid,     # ← 链式指针
            "isSidechain": False,
            "timestamp": ..., "cwd": ..., "sessionId": ..., "version": ...,
            "gitBranch": self._branch,        # 记录当时所在分支
            ...
        }
        if role == "user":
            event["promptId"] = str(_uuid())  # 每条用户输入有独立 promptId
        self._write(event)
        self._prev_uuid = msg_uuid
        return msg_uuid

    def _write(self, event: dict) -> None:    # 整个类只有这 3 行碰文件
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, ensure_ascii=False) + "\n")
```

**uuid 链**（`prev_uuid → uuid`）让消息构成单向链表——即使文件中间坏了几行，也能从任一完好行**回溯整条链**验证完整性。

### 6.5.2　中断恢复

```python
def load_session_messages(path):
    message_types = {"user", "assistant", "system", "tool_result"}
    events = []
    for line in open(path, encoding="utf-8"):
        try:
            ev = json.loads(line)
        except json.JSONDecodeError:      # 残行：静默跳过
            continue
        if ev.get("type") in message_types:
            events.append(ev)
    return events

def list_sessions(directory):
    # glob("*.jsonl") → 按 mtime 倒序，返回 (sid, mtime, count)
```

恢复定位策略：**按文件修改时间倒序**——最近修改的会话最可能是"被打断的那个"。

恢复后的 3 个增强步骤：

1. **file_history_replay**：重放文件历史快照，还原编辑现场；
2. **compaction_replay**：重放压缩事件，重建压缩后的视图；
3. **新提示词**：注入"恢复上下文"提示词，明确告知"你是从断点续跑的"。

### 6.5.3　三步唤醒流程

恢复会话后，Agent 按固定三步重建工作状态：

1. **确认工作目录**（pwd / ls，我在哪）；
2. **查 git 历史**（log / status，我做了什么）；
3. **检查任务进度文件**（TODO / plan，我还要做什么）。

**表 6-8　plain vs three_step 恢复质量（书中实测）**

| 方案 | Phase A 均值 / 方差 | Phase B 均值 / 方差 |
|---|---|---|
| plain（直接续跑） | 1.88 / 1.54 | 2.62 / 0.99 |
| three_step（三步唤醒） | **4.00 / 0.00** | **1.88 / 0.60** |

**解读**：three_step 用少量产出（Phase B 略低）换来**方差为 0 的稳定性**——plain 模式下恢复成功率随机波动大（方差 1.54），偶尔直接卡死；三步唤醒把"恢复质量"从碰运气变成确定性流程。**确定性的小步，胜过不确定的大步。**

---

## 全章总结

本章把"记忆"拆成了闭环的四个部件：

1. **窗口内**：四级火箭分级压缩（L0 截断 → L1 缩略 → L2 摘要 → L3 应急）+ 紧凑指令防目标丢失；
2. **预算**：TokenBudget 五区分配 + 0.70/0.85/0.95 压力阈值决策链；
3. **窗口外**：六层记忆架构 + MEMORY.md 页表模式 + Dream 离线整理；
4. **磁盘上**：JSONL 追加日志 + uuid 链 + 断点恢复三步唤醒。

> 下一章预告：第 7 章将讨论信息验证——Agent 不仅要有记忆，还要能核实记忆与工具结果的真实性。
